## Label Scenario 1

In [1]:
#
# Cell 1: load Libraries
#

import pandas as pd
import numpy as np
import wfdb
import ast

In [3]:
#
# Cell 2: Path for input data and definition of sampling rate
#

#path = 'C:/Users/Buchh/OneDrive/Leuphana_Data_Science/Masterarbeit/ECG/Python/Daten_einlesen/test_data/'
path = 'E:/MA_Daten_2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
sampling_rate=100


In [5]:
#
# Cell 3: Definition of functions
#

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

def aggregate_form_rhythm(y_dic): #BuK
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].form_rhythm)
    return list(set(tmp))


In [7]:
#
# Cell 3a: Specify the directory name for loading of data
#

from pathlib import Path

directory_path = Path("D:/ECG/X_zero")
#directory_path_augm = Path("d:/ECG/X_mixed")

In [9]:
#
# Cell 4:Loading stored raw data, if available. 
#

Y = pd.read_pickle(directory_path/'Y_data.pkl')

In [11]:
#
# Cell 5: FORM AND RHYTHM LABELS
# Load scp_statements.csv for form and rhythm aggregation (BuK)
#

agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[(agg_df.form == 1) | (agg_df.rhythm == 1)]
agg_df['form_rhythm'] = agg_df.index



In [13]:
#
#Cell 6: Apply form and rhythm class, BuK
#

Y['label'] = Y.scp_codes.apply(aggregate_form_rhythm)
display(Y)



,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,label
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,"[LVOLT, SR]"
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,[SBRAD]
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,[SR]
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,[SR]
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,[SR]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21794,21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,...,NaN,", alles,",NaN,NaN,1ES,NaN,7,records100/21000/21833_lr,records500/21000/21833_hr,"[VCLVH, PVC, STACH, NDT]"
21795,21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,...,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr,"[ABQRS, SR]"
21796,21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,...,NaN,", I-AVR,",NaN,NaN,NaN,NaN,2,records100/21000/21835_lr,records500/21000/21835_hr,[SR]
21797,21836,8873.0,64.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-09 18:21:49,...,NaN,NaN,NaN,NaN,SVES,NaN,8,records100/21000/21836_lr,records500/21000/21836_hr,[SR]


In [15]:
#
#Cell 7: Make a copy from Y for further manipulation
#

Y_copy = Y[['ecg_id','patient_id','strat_fold','label']].copy() 
Y_copy

,ecg_id,patient_id,strat_fold,label
0,1,15709.0,3,"[LVOLT, SR]"
1,2,13243.0,2,[SBRAD]
2,3,20372.0,5,[SR]
3,4,17014.0,3,[SR]
4,5,17448.0,4,[SR]
...,...,...,...,...
21794,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]"
21795,21834,20703.0,4,"[ABQRS, SR]"
21796,21835,19311.0,2,[SR]
21797,21836,8873.0,8,[SR]


## Y-data manipulation for augmentation starts 

In [17]:
#
# Cell 7a: choose the path from which augmented X-data should be loaded 
#

from pathlib import Path
directory_path_augm = Path("d:/ECG/X_augmented_1000")
#directory_path_augm = Path("d:/ECG/X_augmented_800")
#directory_path_augm = Path("d:/ECG/X_augmented_600")
#directory_path_augm = Path("d:/ECG/X_augmented_400")
#directory_path_augm = Path("d:/ECG/X_augmented_200")
#directory_path_augm = Path("d:/ECG/X_mixed_augmented")


In [19]:
#
# Cell 7b: loads dicitionary with augmented X-xata
#

import pickle

with open(directory_path_augm/'X_augmented_accomp.pkl', 'rb') as f:
    X_augmented_accomp = pickle.load(f) # deserialize using load()
f.close()  

X_augmented_accomp

{'# of additional X-Data concatenated': 2}

In [21]:
#
# Cell 7c: select a certain row for later control purposes:
#

Y_21794 = Y_copy[Y_copy.index==21794].copy()
Y_21794

,ecg_id,patient_id,strat_fold,label
21794,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]"


In [23]:
#
# Cell 7d: concatenates the df Y_copy according to the value of  
# '# of additional X-Data concatenated':

Y_add=Y_copy
for i in range(0, X_augmented_accomp.get('# of additional X-Data concatenated')):
    Y_copy= pd.concat([Y_copy,Y_add], axis=0, ignore_index= True).copy()

Y_copy    

,ecg_id,patient_id,strat_fold,label
0,1,15709.0,3,"[LVOLT, SR]"
1,2,13243.0,2,[SBRAD]
2,3,20372.0,5,[SR]
3,4,17014.0,3,[SR]
4,5,17448.0,4,[SR]
...,...,...,...,...
65392,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]"
65393,21834,20703.0,4,"[ABQRS, SR]"
65394,21835,19311.0,2,[SR]
65395,21836,8873.0,8,[SR]


In [25]:
#
# Cell 7e: compare two df extracts for control of correct index assignment  
# 21749 + 21799 = 43593

Y_43593 = Y_copy[Y_copy.index==43593].copy()
display(Y_43593)
display(Y_21794)

,ecg_id,patient_id,strat_fold,label
43593,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]"


,ecg_id,patient_id,strat_fold,label
21794,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]"


In [103]:
Y_65392 = Y_copy[Y_copy.index==65392].copy()
display(Y_65392)

,ecg_id,patient_id,strat_fold,label
65392,21833,17180.0,7,"[STACH, VCLVH, PVC, NDT]"


## Y-data manipulation for augmentation ends
## The remaining cells are indentical with the cells of the modul for not augmented data   

In [27]:
#
# Cell 8: in all combinations of labels with "SR" remove "SR" 
# clean list of labels is now stored in the column "codes"
#

Y_copy["codes"] = Y_copy["label"].apply(
    lambda lst: ["SR"] if len(lst) == 1 and lst[0] == "SR"
    else sorted([c for c in lst if c != "SR"])
)
Y_copy

,ecg_id,patient_id,strat_fold,label,codes
0,1,15709.0,3,"[LVOLT, SR]",[LVOLT]
1,2,13243.0,2,[SBRAD],[SBRAD]
2,3,20372.0,5,[SR],[SR]
3,4,17014.0,3,[SR],[SR]
4,5,17448.0,4,[SR],[SR]
...,...,...,...,...,...
65392,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]","[NDT, PVC, STACH, VCLVH]"
65393,21834,20703.0,4,"[ABQRS, SR]",[ABQRS]
65394,21835,19311.0,2,[SR],[SR]
65395,21836,8873.0,8,[SR],[SR]


In [31]:
#
# Cell 9: count the items in the lists in column "codes" and write to "len" column 
#

Y_copy["len"] = Y_copy["codes"].apply(
    lambda lst: len(lst)
)
Y_copy


,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",[LVOLT],1
1,2,13243.0,2,[SBRAD],[SBRAD],1
2,3,20372.0,5,[SR],[SR],1
3,4,17014.0,3,[SR],[SR],1
4,5,17448.0,4,[SR],[SR],1
...,...,...,...,...,...,...
65392,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]","[NDT, PVC, STACH, VCLVH]",4
65393,21834,20703.0,4,"[ABQRS, SR]",[ABQRS],1
65394,21835,19311.0,2,[SR],[SR],1
65395,21836,8873.0,8,[SR],[SR],1


In [33]:
#
# Cell 10: Removes from "Y_copy" all rows resp. ECGs with empty codes
#

Y_copy=Y_copy[Y_copy['len']>0].copy() 
Y_copy

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",[LVOLT],1
1,2,13243.0,2,[SBRAD],[SBRAD],1
2,3,20372.0,5,[SR],[SR],1
3,4,17014.0,3,[SR],[SR],1
4,5,17448.0,4,[SR],[SR],1
...,...,...,...,...,...,...
65392,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]","[NDT, PVC, STACH, VCLVH]",4
65393,21834,20703.0,4,"[ABQRS, SR]",[ABQRS],1
65394,21835,19311.0,2,[SR],[SR],1
65395,21836,8873.0,8,[SR],[SR],1


In [35]:
#
# Cell 11: - convert list element of column "codes" into objects
#

Y_copy['codes'] = Y_copy['codes'].apply(lambda x: ", ".join(x)).copy()
Y_copy

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
2,3,20372.0,5,[SR],SR,1
3,4,17014.0,3,[SR],SR,1
4,5,17448.0,4,[SR],SR,1
...,...,...,...,...,...,...
65392,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]","NDT, PVC, STACH, VCLVH",4
65393,21834,20703.0,4,"[ABQRS, SR]",ABQRS,1
65394,21835,19311.0,2,[SR],SR,1
65395,21836,8873.0,8,[SR],SR,1


In [37]:
#
# Cell 12: Separates the ECGs with SR label.
#

Y_SR = Y_copy[Y_copy['codes'].str.contains(r'^(?=.*SR)')].copy()
Y_SR

,ecg_id,patient_id,strat_fold,label,codes,len
2,3,20372.0,5,[SR],SR,1
3,4,17014.0,3,[SR],SR,1
4,5,17448.0,4,[SR],SR,1
5,6,19005.0,4,[SR],SR,1
6,7,16193.0,7,[SR],SR,1
...,...,...,...,...,...,...
65390,21831,11905.0,9,[SR],SR,1
65391,21832,7954.0,7,[SR],SR,1
65394,21835,19311.0,2,[SR],SR,1
65395,21836,8873.0,8,[SR],SR,1


In [39]:
#
# Cell 13: Substract the not wanted SR ECGs from Y_copy (all ECGs)
#

Y_all_other = Y_copy[~Y_copy.index.isin(Y_SR.index)]
Y_all_other

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1
10,11,11243.0,5,[SARRH],SARRH,1
11,12,11031.0,8,[SBRAD],SBRAD,1
...,...,...,...,...,...,...
65386,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2
65387,21828,13862.0,5,"[PVC, ABQRS, SR]","ABQRS, PVC",2
65388,21829,14508.0,3,"[ABQRS, SR]",ABQRS,1
65392,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]","NDT, PVC, STACH, VCLVH",4


In [43]:
#
# Cell 14: puts dataframes into dictionary
#

all_data_dfs = {'SR':Y_SR, 'all_other':Y_all_other}
all_data_dfs['all_other']#'SR'

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1
10,11,11243.0,5,[SARRH],SARRH,1
11,12,11031.0,8,[SBRAD],SBRAD,1
...,...,...,...,...,...,...
65386,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2
65387,21828,13862.0,5,"[PVC, ABQRS, SR]","ABQRS, PVC",2
65388,21829,14508.0,3,"[ABQRS, SR]",ABQRS,1
65392,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]","NDT, PVC, STACH, VCLVH",4


In [45]:
#
# Cell 15: Inserts a new colum in the dataframes with the simplified labels
# resp. codes
#

for name, df in all_data_dfs.items():
    df = df.copy()
    df.loc[:, 'codes_2'] = name
    all_data_dfs[name] = df

all_data_dfs['all_other']

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1,all_other
1,2,13243.0,2,[SBRAD],SBRAD,1,all_other
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1,all_other
10,11,11243.0,5,[SARRH],SARRH,1,all_other
11,12,11031.0,8,[SBRAD],SBRAD,1,all_other
...,...,...,...,...,...,...,...
65386,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2,all_other
65387,21828,13862.0,5,"[PVC, ABQRS, SR]","ABQRS, PVC",2,all_other
65388,21829,14508.0,3,"[ABQRS, SR]",ABQRS,1,all_other
65392,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]","NDT, PVC, STACH, VCLVH",4,all_other


In [49]:
#
# Cell 16: Selects from the dataframes in all_data_dfs all the instances
# with strat_fold == 10 an put them into separate dataframe
#

test_dfs = {name: df[df['strat_fold'] == 10].copy() for name, df in all_data_dfs.items()}
test_dfs['all_other']#'SR'

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
62,63,15265.0,10,"[ABQRS, SR]",ABQRS,1,all_other
75,76,21214.0,10,[SARRH],SARRH,1,all_other
99,100,21404.0,10,[SARRH],SARRH,1,all_other
115,116,19695.0,10,"[NDT, SR]",NDT,1,all_other
121,122,19832.0,10,[SARRH],SARRH,1,all_other
...,...,...,...,...,...,...,...
65355,21796,17746.0,10,"[ABQRS, AFIB]","ABQRS, AFIB",2,all_other
65368,21809,12931.0,10,"[ABQRS, SR]",ABQRS,1,all_other
65371,21812,20789.0,10,"[SVARR, ABQRS]","ABQRS, SVARR",2,all_other
65377,21818,19204.0,10,"[VCLVH, ABQRS, SR]","ABQRS, VCLVH",2,all_other


In [53]:
#
# Cell 17: substracts for all classes the corresponding test instances from the sets.
# Generates the dictionary "residue_dfs" with all the residue df of the individual classes  
#

residue_dfs = {name: df[~df.index.isin(test_dfs[name].index)].copy() for name, df in all_data_dfs.items()}

residue_dfs['all_other']#'SR'

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
0,1,15709.0,3,"[LVOLT, SR]",LVOLT,1,all_other
1,2,13243.0,2,[SBRAD],SBRAD,1,all_other
7,8,11275.0,9,"[ABQRS, SR]",ABQRS,1,all_other
10,11,11243.0,5,[SARRH],SARRH,1,all_other
11,12,11031.0,8,[SBRAD],SBRAD,1,all_other
...,...,...,...,...,...,...,...
65386,21827,13862.0,5,"[SVARR, ABQRS]","ABQRS, SVARR",2,all_other
65387,21828,13862.0,5,"[PVC, ABQRS, SR]","ABQRS, PVC",2,all_other
65388,21829,14508.0,3,"[ABQRS, SR]",ABQRS,1,all_other
65392,21833,17180.0,7,"[VCLVH, PVC, STACH, NDT]","NDT, PVC, STACH, VCLVH",4,all_other


In [55]:
#
# Cell 18: splits the df in the dictionary "residue_dfs" in training and validation sets.
#

#
# Choosen random states for assessing the influence of the differrent randomly splitted
# training and validation (and test): 42 (first), 31 (second), 7 (third) 
#



training_dfs = {
    name: df.sample(frac=0.75, random_state=7).copy()
    for name, df in residue_dfs.items()
}


validation_dfs = {
    name.replace('_training', '_validation'): df.drop(training_dfs[name].index).copy()
    for name, df in residue_dfs.items()
}


display(training_dfs['all_other'])

print(len(training_dfs['SR']))
print(len(training_dfs['all_other']))

print(len(validation_dfs['SR']))
print(len(validation_dfs['all_other']))

,ecg_id,patient_id,strat_fold,label,codes,len,codes_2
22816,1030,2004.0,9,"[NDT, SR]",NDT,1,all_other
4627,4647,531.0,9,"[PVC, SR]",PVC,1,all_other
42501,20741,10089.0,5,"[ABQRS, SR]",ABQRS,1,all_other
11121,11148,10095.0,5,"[ABQRS, SR]",ABQRS,1,all_other
6578,6599,1047.0,1,"[VCLVH, SBRAD, STD_, NST_]","NST_, SBRAD, STD_, VCLVH",4,all_other
...,...,...,...,...,...,...,...
15722,15760,14522.0,5,"[SR, NST_]",NST_,1,all_other
23678,1892,2548.0,1,"[VCLVH, STD_, SR]","STD_, VCLVH",2,all_other
17409,17447,10013.0,2,[STACH],STACH,1,all_other
2802,2817,7494.0,6,"[SBRAD, LOWT, STD_]","LOWT, SBRAD, STD_",3,all_other


20502
22633
6834
7544


In [57]:
#
# Cell 19: count the number of instances splits the df in 
# the dictionary "residue_dfs" in training and validation sets.
#

dfs_all= {1: test_dfs,
          2: training_dfs,
          3: validation_dfs
         }

summation=sum(df.shape[0] for df in dfs_all.values() for df in df.values())
print('Total sum of all instances in the different sets: ',summation)

Total sum of all instances in the different sets:  63891


In [59]:
#
# Cell 20: combine all test / validation / training sets of all sets (all classes built)
#

#
# Choosen random states for assessing the influence of the differrent randomly splitted
# training and validation (and test): 42 (first), 31 (second), 7 (third) 
#

train_df = pd.concat(training_dfs.values(), axis=0).sample(frac=1, random_state=7)
val_df   = pd.concat(validation_dfs.values(), axis=0).sample(frac=1, random_state=7)
test_df  = pd.concat(test_dfs.values(), axis=0).sample(frac=1, random_state=7)
train_df
print(len(train_df))
print(len(val_df))
print(len(test_df))

43135
14378
6378


In [73]:
#
# Cell 21: 
#

Y_final_dfs = {'train_df' : train_df, 'val_df': val_df, 'test_df' : test_df}

## Following cell (22 to 26) are for preparing X data without augmentation

In [45]:
#
#Cell 22: loads original X values (no augmentation) 
#

from pathlib import Path

X_directory_path = Path("D:/ECG/X_zero")
X = np.load(X_directory_path/'X_float32.npy', mmap_mode='r')


#X_directory_path = Path("D:/ECG/X_mixed")
#X = np.load(X_directory_path/'X_mixed.npz.npy', mmap_mode='r')

In [63]:
#
# Cell 23: divides X values in training-, validation- and test data.
#

X_train = X[train_df.index.tolist(), :, :]
X_val   = X[val_df.index.tolist(), :, :]
X_test  = X[test_df.index.tolist(), :, :]

In [49]:
#
# Cell 24: creates a folder for data storage.
#

# Specify the directory name
#directory_path = Path("D:/ECG/XY_diff_class_0_mixed")
directory_path = Path("D:/ECG/XY_diff_class_0")

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\XY_diff_class_0' already exists.


In [65]:
#
# Cell 25: stores the X data (numpy arrays) to archive in the created folder    
#

np.savez(directory_path/'X_np_arrays_3.npz', a = X_train, b = X_val, c = X_test)

In [67]:
#
# Cell 26: Saves Y_final_dfs and X_final_dfs
#

import pickle

with open(directory_path/'Y_data_3.pkl', 'wb') as f:  # open a text file
    pickle.dump(Y_final_dfs, f) # serialize the list
    f.close()

#with open(directory_path/'X_data.pkl', 'wb') as f:  # open a text file
#    pickle.dump(X_final_dfs, f) # serialize the list
#    f.close()

## The following cells must be executed if one wants to use the augmented datasets of X 

In [75]:
#
# Cell 27: path to read input data
#

#X_directory_path = Path("D:/ECG/X_augmented_1000") # chose the right path beforhand
#X_directory_path = Path("D:/ECG/X_augmented_800") # chose the right path beforhand
#X_directory_path = Path("D:/ECG/X_augmented_600") # chose the right path beforhand
#X_directory_path = Path("D:/ECG/X_augmented_400") # chose the right path beforhand
#X_directory_path = Path("D:/ECG/X_augmented_200") # chose the right path beforhand
X_directory_path = Path("D:/ECG/X_mixed_augmented") # chose the right path beforhand
#X = np.load(X_directory_path/'X_augmented.npz.npy', mmap_mode='r')
X = np.load(X_directory_path/'X_mixed_augmented.npz.npy', mmap_mode='r')

In [77]:
#
# Cell 28: divides X values in training-, validation- and test data.
#

X_train = X[train_df.index.tolist(), :, :]
X_val   = X[val_df.index.tolist(), :, :]
X_test  = X[test_df.index.tolist(), :, :]

In [61]:
#
# Cell 29: creates a folder for data storage.
#

# Specify the directory name
#directory_path = Path("D:/ECG/XY_diff_class_0_aug_1000") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0_aug_800") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0_aug_600") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0_aug_400") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0_aug_200") # chose the right path beforhand
directory_path = Path("D:/ECG/XY_diff_class_0_aug_mixed") # chose the right path beforhand

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\XY_diff_class_0_aug_mixed' already exists.


In [79]:
#
# Cell 30: stores the X data (numpy arrays) to archive in the created folder    
#

np.savez(directory_path/'X_np_arrays_3.npz', a = X_train, b = X_val, c = X_test)

In [81]:
#
# Cell 31: Saves Y_final_dfs and X_final_dfs
#

import pickle

with open(directory_path/'Y_data_3.pkl', 'wb') as f:  # open a text file
    pickle.dump(Y_final_dfs, f) # serialize the list
    f.close()

#with open(directory_path/'X_data.pkl', 'wb') as f:  # open a text file
#    pickle.dump(X_final_dfs, f) # serialize the list
#    f.close()